# Lot D · Gold sur Kaggle : abstention du LLM et LLM-juge

Deux mesures du palier Gold qui demandent le GPU, toutes deux sur le **dev** (test gelé jamais lu) :

1. **Abstention** : le LLM (configuration finale du Silver) sur les 42 variantes du dev sans contraintes ou sans données (`generation/jeux/abstention_dev.jsonl`). Bonne réponse : « Non précisé dans le document », jamais une valeur inventée.
2. **LLM-juge** : Qwen2.5-7B-Instruct 4 bits note les 40 fiches figées (`generation/jeux/relecture_gold_fiches.jsonl`) avec la grille de relecture ; la calibration contre les notes d'Ali et Boutheina se fait ensuite en local.

**Réglages** : Accelerator **GPU T4 x2** (un seul utilisé) ; Internet **On**. Sections 1 à 3 à la main, puis **Save & Run All**.

## 1. Un seul GPU, récupérer la branche `lot-d-gold`

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
%cd /kaggle/working
if os.path.isdir('fardi-caseforge/.git'):
    !git -C fardi-caseforge pull -q origin lot-d-gold
else:
    !git clone -q -b lot-d-gold https://github.com/onsalouini/fardi-caseforge.git
%cd /kaggle/working/fardi-caseforge
!git log --oneline -1
!nvidia-smi -L

## 2. Dépendances

In [ ]:
!pip install -q -U "transformers>=4.44" accelerate bitsandbytes

## 3. Essai rapide : 2 variantes
Chaque ligne doit afficher `essais 1` et `repli False`.

In [ ]:
!python generation/llm.py --docs generation/jeux/abstention_dev.jsonl --abstention --graine 42 --limite 2 --out generation/sorties/essai.jsonl

## 4. Abstention : 42 variantes (~30 min)
Déjà faite le 22/11 : à sauter si l'on relance seulement le juge. `llm.py` applique désormais la règle « Relevés » (Gold) ; sur les réponses du 22/11 rejouées sans GPU : 40 / 42 abstentions correctes, 0 invention.

In [ ]:
!python generation/llm.py --docs generation/jeux/abstention_dev.jsonl --abstention --graine 42 --reprendre --out generation/sorties/llm_abstention_dev_s42.jsonl
!python generation/abstention.py --gen generation/sorties/llm_abstention_dev_s42.jsonl --out reports/lot_d_gold_abstention.md

## 5. LLM-juge : 40 fiches (~15 min)

Consigne v2 (échelle explicite pour chaque critère) : la v1 du 22/11 notait « invention » 0 sur toutes les fiches. Pour relancer seulement le juge : cellules 1, 2, 5 et 6.

In [ ]:
!python generation/juge.py juger --juge llm --out generation/sorties/juge_llm_dev.jsonl
!python generation/juge.py juger --juge regles --out generation/sorties/juge_regles_dev.jsonl

## 5 bis. RQ2 : génération directe par le LLM, 3 graines (~45 min)

Le LLM **rédige** la fiche (schéma `direct`, sans contrôle d'ancrage), mêmes exemples du train. Comparée en local au pipeline Gold (`python generation/rq2.py`). Pour lancer seulement la RQ2 : cellules 1, 2, 5 bis et 6.

In [ ]:
for g in (13, 42, 2026):
    !python generation/llm.py --split dev --schema direct --sans-controle --graine {g} --reprendre --out generation/sorties/rq2_direct_dev_s{g}.jsonl

## 5 ter. Textes libres v4 : rédaction de 40 documents par Qwen (~40 min)

Décision d'équipe du 22/11 : rédaction aidée par Qwen2.5-7B, déclarée dans la carte de données, puis vérification humaine des annotations. Valeurs fictives tirées des lexiques v4 (disjoints du test). Pour lancer seulement cette étape : cellules 1, 2, 5 ter et 6. Le zip contient `libres_v4/` : à décompresser dans `data/gold/`.

In [ ]:
!python generation/textes_libres.py rediger --limite 2
!python generation/textes_libres.py rediger
import shutil
shutil.make_archive('/kaggle/working/libres_v4', 'zip', 'data/gold', 'libres_v4')

## 6. Zip (`sorties_lot_d_gold.zip` dans Output)
À décompresser en local dans `generation/sorties/`, puis `python generation/juge.py calibrer --notes generation/sorties/juge_regles_dev.jsonl generation/sorties/juge_llm_dev.jsonl` quand les deux feuilles de relecture sont remplies.

In [ ]:
import os, shutil
from IPython.display import FileLink
if os.path.exists('reports/lot_d_gold_abstention.md'):
    shutil.copy('reports/lot_d_gold_abstention.md', 'generation/sorties/')
shutil.make_archive('/kaggle/working/sorties_lot_d_gold', 'zip', 'generation', 'sorties')
FileLink('../sorties_lot_d_gold.zip')